# 07 · Rango sugerido de precios (regresión cuantílica)

El tablero muestra, además del precio de referencia, un **rango sugerido**. Se obtiene entrenando dos modelos LightGBM con pérdida cuantílica: uno estima el percentil inferior del precio y otro el superior para un alojamiento con esas características.

Se evalúa con septiembre, solo en las localidades habilitadas, y se registran:

| Métrica | Qué mide | Meta |
|---|---|---|
| `cobertura_nuevos` | % de precios reales de anuncios nuevos dentro del rango | ≈ 80 % |
| `cobertura` | Lo mismo sobre toda la captura de septiembre | ≈ 80 % |
| `ancho_mediano_cop` | Ancho típico del rango en pesos | Lo más angosto posible |
| `razon_sup_inf` | Cuántas veces el límite superior es el inferior | Lo más bajo posible |
| `desvio_meta` | Distancia entre `cobertura_nuevos` y 80 % | 0 |

Un buen rango cubre cerca del 80 % de los precios **sin ser tan ancho que deje de ser útil** para el anfitrión. Las corridas van al experimento `bogoprice-intervalos` de MLflow.

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd

from bogoprice import variables_v2
from bogoprice.experimentos import preparar_datos
from bogoprice.metrics import cobertura
from bogoprice.tracking import iniciar_mlflow, registrar_corrida

# Tag de autor de MLflow
AUTOR = "Edgar"
iniciar_mlflow("bogoprice-intervalos")
datos = preparar_datos()
resultados = []


def intervalo(nombre, version, a_inf, a_sup, registrar=True, **p):
    """Entrena los modelos del percentil inferior y superior, evalúa en septiembre
    y registra la corrida en MLflow. Devuelve las métricas."""
    # Valores base de los dos modelos cuantílicos
    base = {"n_estimators": 600, "learning_rate": 0.03, "num_leaves": 31,
            "min_child_samples": 30, "subsample": 0.8, "subsample_freq": 1,
            "colsample_bytree": 0.7}
    params = base | p
    cols = variables_v2.columnas(version)

    def modelo(alpha):
        return variables_v2.construir_pipeline(
            lgb.LGBMRegressor(objective="quantile", alpha=alpha, verbose=-1, random_state=0, **params),
            version)

    q_inf = modelo(a_inf).fit(datos.train[cols], datos.y_train)
    q_sup = modelo(a_sup).fit(datos.train[cols], datos.y_train)
    lim_inf = np.exp(q_inf.predict(datos.test[cols]))
    lim_sup = np.exp(q_sup.predict(datos.test[cols]))
    y, nuevo = datos.y_test_cop, datos.es_nuevo

    m = {
        "cobertura": cobertura(y, lim_inf, lim_sup),
        "cobertura_nuevos": cobertura(y[nuevo], lim_inf[nuevo], lim_sup[nuevo]),
        "ancho_mediano_cop": float(np.median(lim_sup - lim_inf)),
        "razon_sup_inf": float(np.median(lim_sup / lim_inf)),
    }
    m["desvio_meta"] = abs(m["cobertura_nuevos"] - 80)
    if registrar:
        registrar_corrida(nombre,
                          {"modelo": "LightGBM cuantílico", "conjunto": version,
                           "alpha_inf": a_inf, "alpha_sup": a_sup, **params,
                           "parametro_variado": p and next(iter(p)) or "percentiles"},
                          m, tags={"autor": AUTOR, "conjunto": version, "modelo": "Intervalo"})
    return m

## 1. Percentiles del rango

Con los hiperparámetros base, se prueban **4 pares de percentiles**. Mientras más separados, más cobertura pero más ancho el rango.

In [ ]:
percentiles = [(0.10, 0.90), (0.075, 0.925), (0.05, 0.95), (0.025, 0.975)]

for version in ["v2", "v2_geo"]:
    for a_inf, a_sup in percentiles:
        etiqueta = f"P{a_inf * 100:g}-P{a_sup * 100:g}"
        nombre = f"intervalo_{etiqueta}_{version}"
        m = intervalo(nombre, version, a_inf, a_sup)
        resultados.append({"corrida": nombre, "conjunto": version, "parametro": "percentiles",
                           "valor": etiqueta, **m})
        print(f"{version:7s} {etiqueta:12s} cobertura nuevos = {m['cobertura_nuevos']:.1f} %  "
              f"ancho = ${m['ancho_mediano_cop']:,.0f}")

## 2. Un hiperparámetro a la vez (percentiles P7,5–P92,5)

Con el par de percentiles que apunta al 80 %, se prueban **4 valores de cada hiperparámetro** de los modelos cuantílicos, dejando los demás en la base.

| Hiperparámetro | Base | Valores probados |
|---|---|---|
| `n_estimators` | 600 | 200, 400, 1000, 1500 |
| `learning_rate` | 0,03 | 0,01, 0,02, 0,05, 0,1 |
| `num_leaves` | 31 | 15, 63, 127, 255 |
| `min_child_samples` | 30 | 5, 10, 50, 100 |

In [ ]:
A_INF, A_SUP = 0.075, 0.925

pruebas = {
    "n_estimators": [200, 400, 1000, 1500],
    "learning_rate": [0.01, 0.02, 0.05, 0.1],
    "num_leaves": [15, 63, 127, 255],
    "min_child_samples": [5, 10, 50, 100],
}

for version in ["v2", "v2_geo"]:
    for parametro, valores in pruebas.items():
        for valor in valores:
            nombre = f"intervalo_{parametro}{valor}_{version}"
            m = intervalo(nombre, version, A_INF, A_SUP, **{parametro: valor})
            resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                               "valor": valor, **m})
        print(f"{version:7s} {parametro} listo")

## Resumen

Cobertura en anuncios nuevos y ancho del rango para cada prueba.

In [ ]:
tabla = pd.DataFrame(resultados)
tabla.pivot_table(index=["parametro", "valor"], columns="conjunto",
                  values=["cobertura_nuevos", "ancho_mediano_cop"]).round(1)

**Mejores configuraciones:** las más cercanas al 80 % de cobertura en anuncios nuevos y, entre ellas, las de rango más angosto.

In [ ]:
columnas = ["conjunto", "parametro", "valor", "cobertura_nuevos", "cobertura",
            "ancho_mediano_cop", "razon_sup_inf", "desvio_meta"]
(tabla[tabla["desvio_meta"] <= 2]
 .sort_values(["conjunto", "ancho_mediano_cop"])
 .groupby("conjunto").head(5)[columnas].round(2))

## Conclusiones

- _(par de percentiles que logra ≈ 80 % de cobertura en anuncios nuevos)_
- _(ancho típico del rango y si es útil para el anfitrión)_
- _(hiperparámetros que reducen el ancho sin perder cobertura)_
- _(diferencia con y sin coordenadas)_